# Diffusion Model for Image Denoising
This notebook demonstrates how to implement a basic diffusion model for image denoising using PyTorch. Diffusion models are a class of generative models that learn to reverse a gradual noising process, producing high-quality images from random noise.

**Outline:**
1. Import Required Libraries
2. Prepare Dataset
3. Define Diffusion Model Architecture
4. Implement Forward Diffusion Process
5. Implement Reverse (Denoising) Process
6. Train the Diffusion Model
7. Generate Samples with the Trained Model
8. Visualize Generated Samples

In [ ]:
# 1. Import Required Libraries
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
from torchvision import transforms, datasets
from torch.utils.data import DataLoader, Dataset
from PIL import Image
import os
from tqdm import tqdm

## 2. Prepare Dataset
Load and preprocess the image dataset for training the diffusion model. We'll use the clean and noisy images from your existing dataset structure.

In [ ]:
# Load dataset mapping and use only valid pairs
import pandas as pd

mapping_path = '../Dataset/dataset_mapping.csv'
mapping_df = pd.read_csv(mapping_path)
paired_img_names = mapping_df.dropna(subset=['clean', 'noisy'])[['clean', 'noisy']].values.tolist()

class PairedImageDataset(Dataset):
    def __init__(self, clean_dir, noisy_dir, pairs, transform=None):
        self.clean_dir = clean_dir
        self.noisy_dir = noisy_dir
        self.pairs = pairs
        self.transform = transform
    def __len__(self):
        return len(self.pairs)
    def __getitem__(self, idx):
        clean_name, noisy_name = self.pairs[idx]
        clean_path = os.path.join(self.clean_dir, clean_name)
        noisy_path = os.path.join(self.noisy_dir, noisy_name)
        clean_img = Image.open(clean_path).convert('RGB')
        noisy_img = Image.open(noisy_path).convert('RGB')
        if self.transform:
            clean_img = self.transform(clean_img)
            noisy_img = self.transform(noisy_img)
        return noisy_img, clean_img

dataset = PairedImageDataset(clean_dir, noisy_dir, paired_img_names, transform)
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

## 3. Define Diffusion Model Architecture
We will use a simple U-Net architecture as the backbone for the denoising diffusion model.

In [ ]:
# Simple U-Net for diffusion denoising
class UNet(nn.Module):
    def __init__(self, in_channels=3, out_channels=3, features=[64, 128, 256, 512]):
        super(UNet, self).__init__()
        self.downs = nn.ModuleList()
        self.ups = nn.ModuleList()
        # Downsampling
        for feature in features:
            self.downs.append(nn.Sequential(
                nn.Conv2d(in_channels, feature, 3, stride=1, padding=1),
                nn.ReLU(),
                nn.Conv2d(feature, feature, 3, stride=1, padding=1),
                nn.ReLU(),
            ))
            in_channels = feature
        # Upsampling
        for feature in reversed(features):
            self.ups.append(nn.Sequential(
                nn.Conv2d(in_channels, feature, 3, stride=1, padding=1),
                nn.ReLU(),
                nn.Conv2d(feature, feature, 3, stride=1, padding=1),
                nn.ReLU(),
            ))
            in_channels = feature
        self.bottleneck = nn.Sequential(
            nn.Conv2d(features[-1], features[-1]*2, 3, stride=1, padding=1),
            nn.ReLU(),
            nn.Conv2d(features[-1]*2, features[-1], 3, stride=1, padding=1),
            nn.ReLU(),
        )
        self.final_conv = nn.Conv2d(features[0], out_channels, 1)

    def forward(self, x, t):
        # t: timestep embedding (can be added for advanced models)
        skip_connections = []
        for down in self.downs:
            x = down(x)
            skip_connections.append(x)
            x = F.max_pool2d(x, 2)
        x = self.bottleneck(x)
        skip_connections = skip_connections[::-1]
        for idx in range(len(self.ups)):
            x = F.interpolate(x, scale_factor=2, mode='nearest')
            x = self.ups[idx](x)
            if x.shape == skip_connections[idx].shape:
                x = x + skip_connections[idx]
        return self.final_conv(x)

model = UNet().to(device)

## 4. Implement Forward Diffusion Process
The forward process gradually adds noise to the images over multiple timesteps.

In [ ]:
# Forward diffusion process (noise schedule)
def linear_beta_schedule(timesteps):
    beta_start = 0.0001
    beta_end = 0.02
    return torch.linspace(beta_start, beta_end, timesteps)

timesteps = 1000
betas = linear_beta_schedule(timesteps)
alphas = 1. - betas
alpha_cumprod = torch.cumprod(alphas, dim=0)

def forward_diffusion_sample(x_0, t, device='cpu'):
    noise = torch.randn_like(x_0)
    sqrt_alpha_cumprod_t = torch.sqrt(alpha_cumprod[t]).to(device)
    sqrt_one_minus_alpha_cumprod_t = torch.sqrt(1 - alpha_cumprod[t]).to(device)
    return sqrt_alpha_cumprod_t * x_0 + sqrt_one_minus_alpha_cumprod_t * noise, noise

## 5. Implement Reverse (Denoising) Process
The reverse process uses the model to predict and remove noise from the images, reconstructing the original image from noisy inputs.

In [ ]:
# Reverse (denoising) process
@torch.no_grad()
def sample_denoised_image(model, img_shape, timesteps, device):
    x = torch.randn(img_shape).to(device)
    for t in reversed(range(timesteps)):
        t_tensor = torch.tensor([t], dtype=torch.long).to(device)
        predicted_noise = model(x, t_tensor)
        alpha = alphas[t]
        alpha_cumprod_t = alpha_cumprod[t]
        beta = betas[t]
        if t > 0:
            noise = torch.randn_like(x)
        else:
            noise = torch.zeros_like(x)
        x = (1 / torch.sqrt(alpha)) * (x - ((1 - alpha) / torch.sqrt(1 - alpha_cumprod_t)) * predicted_noise) + torch.sqrt(beta) * noise
    return x

## 6. Train the Diffusion Model
Train the model to predict the noise added at each timestep.

In [ ]:
# Training loop
epochs = 5  # For demonstration, increase for real training
optimizer = optim.Adam(model.parameters(), lr=2e-4)
loss_fn = nn.MSELoss()

for epoch in range(epochs):
    model.train()
    pbar = tqdm(dataloader, desc=f'Epoch {epoch+1}/{epochs}')
    for noisy_imgs, clean_imgs in pbar:
        noisy_imgs = noisy_imgs.to(device)
        clean_imgs = clean_imgs.to(device)
        t = torch.randint(0, timesteps, (noisy_imgs.size(0),), device=device).long()
        x_t, noise = forward_diffusion_sample(clean_imgs, t, device=device)
        predicted_noise = model(x_t, t)
        loss = loss_fn(predicted_noise, noise)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        pbar.set_postfix({'loss': loss.item()})

## 7. Generate Samples with the Trained Model
Use the trained model to generate new images by sampling from noise and applying the reverse process.

In [ ]:
# Generate samples
model.eval()
num_samples = 4
img_shape = (num_samples, 3, img_size, img_size)
samples = sample_denoised_image(model, img_shape, timesteps, device)

## 8. Visualize Generated Samples
Display generated images and compare them to real samples for qualitative evaluation.

In [ ]:
# Visualize generated samples
def show_images(images, title='Generated Samples'):
    images = images.detach().cpu()
    images = (images * 0.5) + 0.5  # Denormalize
    fig, axs = plt.subplots(1, images.shape[0], figsize=(15, 5))
    for i in range(images.shape[0]):
        img = images[i].permute(1, 2, 0).numpy()
        axs[i].imshow(np.clip(img, 0, 1))
        axs[i].axis('off')
    plt.suptitle(title)
    plt.show()

show_images(samples)